# Gradient boosting trees
We have seen many ML algorithms, from decision trees to support vector machines. Decision tree models are specially popular for their simplicity and interpretability, however, have serious handycaps like they are prone to overfitting also can't learn complex relationships. Random forests and extremly randomized trees are popular approaches to solve those issues because use an ensemble approach combining multiple "weak learners" (decision trees) to create a "strong model". In the previous post we learned about gradient descent in neural nets, a simple but powerful technique to optimize model parameters from errors. 

## Boosting
Boosting is an ensemble approach for training a strong ML estimator from many weak learners, unlike bagging that focus on training every learner independently and combining their outcomes. Boosting focus on creating new estimators to learn from the errors of the previous ones, the outcome is often a weigthed sum of individual estimators outcome.

## Gradient boosting decision trees
Gradient boosting decision trees is a ML strategy that uses a boosting approach with the descent gradient algorithm to optimize model performance on decision tree estimators. Formerly the model is defined as:

$$
    F(x) = h_{0}(x) + \eta \sum_{m=1}^{M} h_{m}(x)
$$

where:
- $h_{0}(x)$ is the base learner.
- $h_{m}(x)$ is the decision tree added at $m$ iteration.
- $M$ is the number of trees.
- $\eta$ is the learning rate.

### Training
Unlike random forests that uses boostrapping to create a custom training dataset for each learner or extra-trees that trains every tree with the entire dataset but choose random thresholds for sample splits. Gradient based approach is focused on training new estimators to fix the errors of the previous one.

**Algorithm:**

1. Train the base learner $h_0$ as the decision tree that minimizes the loss function $L$ from training samples $x$ and target values $y$. Let's define $y_{0,i} = y_i$
2. For a given estimator $h_k$, calculate the output $h_k(x)$.
3. Measure the error for each sample $x_i$ using $L(y_{k,i} , h_k(x_i))$ loss function.
4. Calculate the residuals $r_{k,i}$ for each sample $i$ for the estimator $k$.

$$
r_{k,i} = -\frac{\partial L(y_{k,i}, h_{k}(x_i))}{\partial h_{k}(x_i)}
$$

5. Train a new estimator $h_{k+1}$ repeating steps 2 to 4 using the residuals of the previous estimators as target. This is: $y_{k+1,i} = r_{k+1,i}$, so our training samples will be represented as:

$$
    (x_i, r_{i,k+1})
$$

6. Repeat the process until reaching the desired number of estimators, the tolerance threshold has been reached or an early stopping criterion is reached.

Each tree is trained as a regular decision tree so there aren't changes in the single estimator training. With this approach, each estimator is defined as a recurrence relationship with the previous trained:

$$
h_{k} = h_{k-1} + \eta h_{k}
$$

but remembering that every estimator but the base is trained to fit the residuals of the previous one trained at previous iteration, so the recurrence becomes:
$$
h_{k} \approx -\frac{\partial L(y_{k,i}, h_{k}(x_i))}{\partial h_{k}(x_i)}
$$
<br />
$$
h_{k} = h_{k-1} - \eta \frac{\partial L(y_{k,i}, h_{k}(x_i))}{\partial h_{k}(x_i)}
$$

## Implementations
### Scikit learn
Scikit offers the basic implementation at `GradientBoostingClassifier` and `GradientBoostingRegressor` classes and also has a more efficient on classes `HistGradientBoostingClassifier` and `HistGradientBoostingRegressor`. The "Hist" variant support missing and categorical values, but their key advantage is the performance gain because it uses discretization on quantitative features using 255 at most bins to acelerate sample split selection instead of calculating median values of each pair of unique values for each feature.

### XGBoost
Probably the most popular, is a ML framework for building gradient boosting models aimed to distributed environments, also implements more efficient variations of the algorithm described here adding regularization terms to avoid overfitting. Even has native support for GPU training and handles missing and categorical features natively. Main classes are `XGBClassifier` and `XGBRegressor`, both follows the Scikit interface, so can work with other Scikit classes like `Pipeline`. XGBoost library is available in Python, Julia, C++, R, among other languages.

## Tasks
### Classification
The definition is very similar to the presented model structure, however it performs well for regression. On classification is more complex because we must make that every estimator $h_k$ depends on $L$ loss function, so we must add a decision function that takes the model outcome and returns a probability.
- **Binary case:** Uses the sigmoid function ($\sigma$) on model output $F(x)$ to estimate the probability of belonging to the positive class. When $\sigma(x) \geq 0.5$ sample $x$ belongs to positive class, otherwise to the negative.
$$
    p(y = 1 \mid x) = \sigma(x) = \frac{1}{1+e^{-F(x)}}
$$
$$
    \hat{y}(x) =
    \begin{cases} 
        1 & \text{if } p(y = 1 \mid x) \geq 0.5 \\
        0 & \text{if } p(y = 1 \mid x) < 0.5
    \end{cases}
$$
- **Multiclass case:** Trains as many models as classes and applies the softmax function. For $K$ classes it fits $F_1(x), F_2(x), ..., F_k(x)$ functions that speciallizes in scoring instances to class $k$ at each iteration. Works like softmax layer in neural nets so ir just selects the class with most probability.

$$
    p(y=k \mid x) = \theta(x, k) =\frac{e^{F_k(x)}}{\sum_{j=1}^{n} e^{F_j(x)}}
$$
<br />
$$
    \hat{y} = \operatorname*{arg\,max}_{c \in \{1,\ldots, classes \}}
\theta(x, k)
$$

However note that even for classification task, the estimator are still regressors, not classifiers because we are training them to predict the gradients (continuous values). Keep in mind that this is a general idea, XGBoost has its custom implementation for instance.

In [ ]:
from pandas import read_csv
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.metrics import accuracy_score, f1_score

data = read_csv("/kaggle/input/datasets/iabhishekofficial/mobile-price-classification/train.csv")
COL_MAPPER = {"fc": "front_camera_pixels", "four_g": "has_4g", "int_memory": "storage_size", 
              "mobile_wt": "weight", "m_dep": "depth_cm", "n_cores": "cpu_cores", "pc": "rear_camera_pixels",
              "px_width": "screen_width_pixels", "px_height": "screen_height_pixels", "sc_w": "screen_width_cm",
              "sc_h": "screen_height_cm", "three_g": "has_3g", "touch_screen": "has_touch_screen", "wifi": "has_wifi",
              "blue": "has_bluetooth", "clock_speed": "cpu_clock_speed"}

renamed_data = data.rename(columns=COL_MAPPER)

NUMS = ["battery_power","cpu_clock_speed", "depth_cm", "front_camera_pixels", "storage_size", "weight", "cpu_cores",
       "rear_camera_pixels", "screen_height_pixels","screen_width_pixels","ram","screen_height_cm","screen_width_cm","talk_time"]
BINS = ["has_4g", "has_3g", "has_bluetooth", "dual_sim", "has_wifi", "has_touch_screen"]
FEATURES = NUMS + BINS
TAG = "price_range"

x_train, x_test, y_train, y_test = train_test_split(renamed_data[FEATURES], renamed_data[TAG], train_size=0.8, random_state=123)

TRANSFORMER = ColumnTransformer([
    ("standarized", StandardScaler(), NUMS)
], remainder="passthrough")

X_train = TRANSFORMER.fit_transform(x_train)
X_test = TRANSFORMER.transform(x_test)

# Model definition
CLASSIFICATION_MODEL = MLPClassifier(hidden_layer_sizes=(20, 15, 10, 5), solver="lbfgs", activation="relu", random_state=123)
CLASSIFICATION_MODEL.fit(X_train, y_train)
y_pred = CLASSIFICATION_MODEL.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
f1Score = f1_score(y_test, y_pred, average="weighted")
print(f"Accuracy: {(accuracy*100):2f} %")
print(f"Weighted F1-Score: {(f1Score*100):2f} %")

### Regression
The described algorithm works for regression tasks, so it just evaluates the input vector $x$ in every decision tree and adds their outputs to build the model outcome $F(x)$ there isn't necessary to use decision function like  classification tasks.

In [ ]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error

data = read_csv("/kaggle/input/datasets/jsonali2003/mobile-price-prediction-dataset/Mobile Price Prediction Datatset.csv")

data = data.drop(columns=["Unnamed: 0"])
COL_MAPPER = {"Brand me": "brand", "Ratings": "user_ratings", "RAM": "ram", 
              "ROM": "storage_size", "Mobile_Size": "display_size_inches", "Primary_Cam": "main_camera_px", "Selfi_Cam": "selfie_camera_px",
              "Battery_Power": "battery", "Price": "price"}
renamed_data = data.rename(columns=COL_MAPPER)

NUMS = ["user_ratings","ram", "storage_size", "display_size_inches", "main_camera_px", "selfie_camera_px", "battery"]
CATS = ["brand"]
FEATURES = NUMS + CATS
TAG = "price"

x_train, x_test, y_train, y_test = train_test_split(renamed_data[FEATURES], renamed_data[TAG], train_size=0.8, random_state=123)

CAT_TRANSFORMER = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
])

TRANSFORMER = ColumnTransformer([
    ("num", KNNImputer(), NUMS),
    ("cat", CAT_TRANSFORMER, CATS)
])

X_train = TRANSFORMER.fit_transform(x_train)
X_test = TRANSFORMER.transform(x_test)

# Model definition
REGRESSION_MODEL = MLPRegressor(hidden_layer_sizes=(8,8,8,4,2), activation="identity", solver="lbfgs", random_state=123)
REGRESSION_MODEL.fit(X_train, y_train)
y_pred = REGRESSION_MODEL.predict(X_test)

squared_error = mean_squared_error(y_test, y_pred)
absolute_error = mean_absolute_error(y_test, y_pred)
print(f"Mean squared error: {squared_error:2f}")
print(f"Mean absolute error: {absolute_error:2f}")